In [1]:
import pandas as pd
import numpy as np
import random
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datetime import datetime, timedelta

In [2]:
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {plt.matplotlib.__version__}")
print(f"Seaborn: {sns.__version__}")

Pandas: 3.0.3
NumPy: 2.4.6
Matplotlib: 3.11.0
Seaborn: 0.13.2


### 1. Dataset e informações básicas

In [ ]:
# criar dataset sintético com 200 registros de vendas e com alguns dados sujos

def gerar_dataset_vendas(n_registros=200, seed=1):
    random.seed(seed)
    np.random.seed(seed)
    produtos = ['Notebook', 'Smartphone', 'Tablet', 'Monitor', 'Teclado', 'Mouse']
    precos = { 'Notebook': 3500, 'Smartphone': 2200, 'Tablet': 1800,
               'Monitor': 1200, 'Teclado': 250, 'Mouse': 120 }
    categorias = { "Notebook": "Computadores", "Smartphone": "Celulares",
                   "Tablet": "Celulares", "Monitor": "Computadores", "Teclado": "Periféricos",
                   "Mouse": "Periféricos" }
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
    clientes = [f"Cliente_{i:03d}" for i in range(1, 31)]
    
    data_inicio = datetime(2024, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        quantidade = random.randint(1, 10)
        preco = precos[produto]
        data = data_inicio + timedelta(days=random.randint(0, 364))

        # Inserindo dados sujos para limpeza
        if random.random() < 0.05:
            quantidade = None 
        if random.random() < 0.04:
            preco = None 
        if random.random() < 0.03:
            produto = " " + produto 
        
        data_str = data.strftime("%Y-%m-%d") if random.random() > 0.02 else "DATA INVALIDA"
        dados.append({
            "id_venda": i + 1,
            "data_venda": data_str,
            "cliente": random.choice(clientes),
            "produto": produto,
            "categoria": categorias.get(produto.strip(), "Outros"),
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco
        })
    
    return pd.DataFrame(dados)

In [4]:
# exibir informações básicas do DataFrame

def inspecionar_dados(df):
    print("\n=== INSPEÇÃO INICIAL DO DATASET ===")
    print(f"Shape: {df.shape}")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados:\n{df.dtypes}")
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    print(f"\nPrimeiros registros:\n{df.head()}")
    print(f"\nEstatísticas descritivas:\n{df.describe()}")
    #return df.describe(include="all")

In [6]:
# Gera o dataset e salva em /data/raw
df_bruto = gerar_dataset_vendas()
os.makedirs('data', exist_ok=True) # Adicionado para garantir que o diretório exista
df_bruto.to_csv("../data/raw/vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")
df_bruto.head() 

Dataset gerado com 200 registros.


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2024-02-02,Cliente_026,Smartphone,Celulares,Sul,10.0,2200.0
1,2,2024-01-15,Cliente_023,Notebook,Computadores,Centro-Oeste,8.0,3500.0
2,3,2024-10-29,Cliente_018,Tablet,Celulares,Sudeste,4.0,1800.0
3,4,2024-08-04,Cliente_018,Monitor,Computadores,Sul,4.0,1200.0
4,5,2024-12-12,Cliente_027,Tablet,Celulares,Norte,4.0,1800.0


In [7]:
# informações básicas do DataFrame
inspecionar_dados(df_bruto)


=== INSPEÇÃO INICIAL DO DATASET ===
Shape: (200, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda            str
cliente               str
produto               str
categoria             str
regiao                str
quantidade        float64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda           0
data_venda         0
cliente            0
produto            0
categoria          0
regiao             0
quantidade        11
preco_unitario     5
dtype: int64

Primeiros registros:
   id_venda  data_venda      cliente     produto     categoria        regiao  \
0         1  2024-02-02  Cliente_026  Smartphone     Celulares           Sul   
1         2  2024-01-15  Cliente_023    Notebook  Computadores  Centro-Oeste   
2         3  2024-10-29  Cliente_018      Tablet     Celulares       Sudeste   
3         4  2024-08-04  Cliente_018     Mon